In [ ]:
%%writefile main.cu

// This program computes a simple version of matrix multiplication
// By: Nick from CoffeeBeforeArch

#include <algorithm>
#include <cassert>
#include <cstdlib>
#include <functional>
#include <iostream>
#include <vector>

using std::cout;
using std::generate;
using std::vector;

#define TILE 16

__global__ void matrixMul(const int *a, const int *b, int *c, int N) {
    size_t row = blockDim.y * blockIdx.y + threadIdx.y;
    size_t col = blockDim.x * blockIdx.x + threadIdx.x;
    
    __shared__ int tile_a[TILE][TILE];
    __shared__ int tile_b[TILE][TILE];
    
    int sum = 0;
    
    for (int i = 0; i < N / TILE; ++i) {
      tile_a[threadIdx.y]threadIdx.x] = 
    }

    c[row * N + col] = sum;
}

// Check result on the CPU
bool verify_result(vector<int> &a, vector<int> &b, vector<int> &c, int N) {
  // For every row...
  for (int i = 0; i < N; i++) {
    // For every column...
    for (int j = 0; j < N; j++) {
      // For every element in the row-column pair
      int tmp = 0;
      for (int k = 0; k < N; k++) {
        // Accumulate the partial results
        tmp += a[i * N + k] * b[k * N + j];
      }

      // Check against the CPU result
      if (tmp != c[i * N + j]) {
        cout << "Mismatch at (" << i << ", " << j << "): " << tmp
             << " != " << c[i * N + j] << std::endl;
        return false;
      }
    }
  }
  return true;
}

int main() {
  // Matrix size of 1024 x 1024;
  int N = 1 << 10;
  
  vector<int> h_a(N * N);
  vector<int> h_b(N * N);
  vector<int> h_c(N * N);

  int *d_a, *d_b, *d_c;
  cudaMalloc(&d_a, N * N * sizeof(int));
  cudaMalloc(&d_b, N * N * sizeof(int));
  cudaMalloc(&d_c, N * N * sizeof(int));
  
  generate(h_a.begin(), h_a.end(), []() { return rand() % 100; });
  generate(h_b.begin(), h_b.end(), []() { return rand() % 100; });
  
  cudaMemcpy(d_a, h_a.data(), N * N * sizeof(int), cudaMemcpyHostToDevice);
  cudaMemcpy(d_b, h_b.data(), N * N * sizeof(int), cudaMemcpyHostToDevice);
  
  size_t threadsPerBlockDim = 16;

  int threadPerBlockDim = 16;
  int blocksPerGridDim = (N + threadPerBlockDim - 1) / threadPerBlockDim;
  
  dim3 grid(blocksPerGridDim, blocksPerGridDim);
  dim3 block(threadPerBlockDim, threadPerBlockDim);

  matrixMul<<<grid, block>>>(d_a, d_b, d_c, N);
  
  cudaMemcpy(h_c.data(), d_c, N * N * sizeof(int), cudaMemcpyDeviceToHost);
  
  bool is_equal = verify_result(h_a, h_b, h_c, N);
  
  if (is_equal) {
    cout << "Results match!" << std::endl;
  } else {
    cout << "Results do not match!" << std::endl;
  }
  
  cudaFree(d_a);
  cudaFree(d_b);
  cudaFree(d_c);
}


Overwriting main.cu


In [10]:
%%shell

nvcc main.cu -o main
time ./main

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
main.cu(71): warning #177-D: variable "threadsPerBlockDim" was declared but never referenced
    size_t threadsPerBlockDim = 16;
           ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

Results match!

real	0m13.359s
user	0m12.885s
sys	0m0.414s
